<h3>Investigar os problemas</h3>


In [0]:
df = spark.table("projeto_churn.bronze.clientes_raw")

# Ver qual o tipo de TotalChanges
df.printSchema()

# Contar os registros problemáticos
df.filter("trim(TotalCharges) = ''").count()

# Checar duplicatas
print(df.count(), df.select("customerID").count())

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _arquivo_origem: string (nul

<h3> Limpar e tipar

In [0]:
from pyspark.sql.functions import col, when, trim, to_date, current_date, expr

df_silver = (df
             # converter TotalCharges para numero, tratando vazios
             .withColumn("TotalCharges",
                         when(trim(col("TotalCharges")) == "", None)
                         .otherwise(trim(col("TotalCharges")).cast("double")))
             
             # padronizar os booleanos textuais
             .withColumn("churn_flag", when(col("Churn") == "Yes", 1).otherwise(0))
             .withColumn("cliente_idoso", when(col("SeniorCitizen") == 1, True).otherwise(False))

             # regra de negócio: cliente novo tem TotalCharges nulo, então preenchemos
             .withColumn("TotalCharges",
                         when(col("TotalCharges").isNull(), col("MonthlyCharges"))
                         .otherwise(col("TotalCharges")))
             
             # enriquecimento: segmentação por tempo de casa
             .withColumn("faixa_tenure",
                         when(col("tenure") <= 6, "0-6 meses")
                         .when(col("tenure") <= 12, "7-12 meses")
                         .when(col("tenure") <= 24, "13-24 meses")
                         .when(col("tenure") <= 48, "25-48 meses")
                         .otherwise("49+ meses"))
             
             .dropDuplicates(["customerID"])
            )


(df_silver.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .option("delta.enableChangeDataFeed", "true")
 .saveAsTable("projeto_churn.silver.clientes"))

<h3> Adicionar validação de qualidade

In [0]:
from pyspark.sql.functions import count, sum as _sum

validacoes = spark.sql("""
                       SELECT
                            COUNT(*) AS total_linhas,
                            SUM(CASE WHEN customerID IS NULL THEN 1 ELSE 0 END) AS ids_nulos,
                            SUM(CASE WHEN MonthlyCharges <= 0 THEN 1 ELSE 0 END) AS cobranca_invalida,
                            SUM(CASE WHEN tenure < 0 THEN 1 ELSE 0 END) AS tenure_negativo,
                            ROUND(AVG(churn_flag) * 100, 2) AS taxa_churn_pct
                        FROM projeto_churn.silver.clientes
                    """)

display(validacoes)

total_linhas,ids_nulos,cobranca_invalida,tenure_negativo,taxa_churn_pct
7043,0,0,0,26.54


<h3> Documentar a tabela

In [0]:
%sql
COMMENT ON TABLE projeto_churn.silver.clientes IS
    'Base de clientes limpa e tipada. Grão: um registro por cliente.';

ALTER TABLE projeto_churn.silver.clientes
    ALTER COLUMN churn_flag COMMENT 'Indicador binário de cancelamento: 1 = cancelou.'